In [9]:
%pip install requests pillow shapely geojson requests pyproj

Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
from PIL import Image
from shapely.geometry import Polygon, Point
from io import BytesIO

headers = {
    'token': '66c60563e408dca80ab9422bd6d1d9b7'
}

item = requests.get('https://api.dataforsyningen.dk/rest/skraafoto_api/v1.0/collections/skraafotos2023/items/2023_82_23_2_0010_00001123', headers=headers)
info_dict = item.json()
img_url = info_dict["properties"]["asset:data"]

poly= info_dict['geometry']['coordinates'][0]
rotation_mat = info_dict['properties']['pers:rotation_matrix']

In [11]:
print(info_dict, rotation_mat)

{'type': 'Feature', 'stac_version': '1.0.0', 'stac_extensions': ['https://stac-extensions.github.io/view/v1.0.0/schema.json', 'https://stac-extensions.github.io/projection/v1.0.0/schema.json', 'https://raw.githubusercontent.com/stac-extensions/perspective-imagery/main/json-schema/schema.json'], 'id': '2023_82_23_2_0010_00001123', 'collection': 'skraafotos2023', 'geometry': {'type': 'Polygon', 'coordinates': [[[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]]}, 'bbox': [9.791136734961823, 55.86958349197774, 9.816435520740168, 55.88267537764091], 'properties': {'datetime': '2023-04-08T13:13:41Z', 'gsd': 0.09, 'license': 'various', 'platform': 'Fixed-wing aircraft', 'instruments': ['Osprey-434S42218X318205-f120_Rev01.00_C114'], 'providers': [{'name': 'field', 'roles': ['producer']}, {'url': 'https://sdfi.dk/', 'name': 'SDFI', 'roles': ['licen

In [12]:


print(img_url, poly)

https://cdn.dataforsyningen.dk/skraafoto_server/COG_oblique_2023/10km_619_55/1km_6192_550/2023_82_23_2_0010_00001123.tif [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]


In [ ]:
import requests
from shapely.geometry import Polygon
import cv2
import numpy as np

local= True

if not local:
    # Load image from CDN
    response = requests.get(img_url)
    image_arr = np.array(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_arr, cv2.IMREAD_COLOR)
else:
    poly = [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]
    image = cv2.imread('/Users/domopapp/Documents/bach/bachelor/playground/skraafoto_images/2023_82_23_2_0010_00001123.tif')
    rotation_matrix_list= [0.9999777436461087, -0.00661694332508786, -0.0008533894011739274, 0.005274316520016631, 0.7057005897569285, 0.7084905498325083, -0.0040858044109012915, -0.7084792824619192, 0.7057197832883532]


img_height, img_width, _ = image.shape
print(img_height, img_width)
polygon = Polygon(poly)


# Convert to 3×3 NumPy array
rotation_matrix = np.array(rotation_matrix_list).reshape(3, 3)

# Load image from CDN

img_height, img_width, _ = image.shape

# Load GeoJSON and extract bounding box
polygon = Polygon(poly)

# Get bounding box of the geojson coordinates
min_lon, min_lat, max_lon, max_lat = polygon.bounds

# Define real-world control points for precise homography
src_pts = np.array([
    [0, 0],  # Top-left (Image pixel coordinates)
    [img_width - 1, 0],  # Top-right
    [img_width - 1, img_height - 1],  # Bottom-right
    [0, img_height - 1]  # Bottom-left
], dtype=np.float32)

dst_pts = np.array([
    [min_lon, max_lat],  # Corresponding Geo-coordinates
    [max_lon, max_lat],
    [max_lon, min_lat],
    [min_lon, min_lat]
], dtype=np.float32)

# Compute homography matrix
H, _ = cv2.findHomography(src_pts, dst_pts, method=cv2.RANSAC)

# Compute inverse homography for validation
H_inv, _ = cv2.findHomography(dst_pts, src_pts, method=cv2.RANSAC)

def apply_rotation_correction(x, y):
    """Corrects pixel coordinates using rotation matrix."""
    pixel_coords = np.array([x, y, 1], dtype=np.float32).T
    corrected_coords = np.dot(rotation_matrix, pixel_coords)
    return corrected_coords[0], corrected_coords[1]

def pixel_to_latlon(x, y):
    """Convert image pixel coordinates to latitude/longitude using rotation and homography."""
    x_corrected, y_corrected = apply_rotation_correction(x, y)
    pixel_coords = np.array([[x_corrected, y_corrected, 1]], dtype=np.float32).T
    geo_coords = np.dot(H, pixel_coords)
    geo_coords /= geo_coords[2]  # Normalize
    lat, lon = geo_coords[1][0], geo_coords[0][0]
    return round(lat, 8), round(lon, 8)

def latlon_to_pixel(lat, lon):
    """Convert latitude/longitude back to image pixel coordinates for validation."""
    geo_coords = np.array([[lon, lat, 1]], dtype=np.float32).T
    pixel_coords = np.dot(H_inv, geo_coords)
    pixel_coords /= pixel_coords[2]  # Normalize
    x, y = int(pixel_coords[0][0]), int(pixel_coords[1][0])
    return x, y

def mouse_callback(event, x, y, flags, param):
    """Handle mouse movement over the image."""
    if event == cv2.EVENT_MOUSEMOVE:
        lat, lon = pixel_to_latlon(x, y)
        x_back, y_back = latlon_to_pixel(lat, lon)  # Validation check
        text = f"Lat: {lat-0.004:.8f}, Lon: {lon:.8f}\nX: {x}, Y: {y} (Back: {x_back}, {y_back})"
        temp_image = image.copy()
        cv2.putText(temp_image, text, (0, 400), cv2.FONT_HERSHEY_SIMPLEX, 6, (0, 255, 0), 12)
        cv2.imshow("Image", temp_image)

# Display image using OpenCV
cv2.imshow("Image", image)
cv2.setMouseCallback("Image", mouse_callback)
cv2.waitKey(0)
cv2.destroyAllWindows()


10560 14144


In [ ]:
import requests
from shapely.geometry import Polygon
import cv2
import numpy as np

local = True

if not local:
    # Load image from CDN
    response = requests.get(img_url)
    image_arr = np.array(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_arr, cv2.IMREAD_COLOR)
else:
    poly = [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471],
            [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553],
            [9.791136734961823, 55.88267537764091]]
    image = cv2.imread('/Users/domopapp/Documents/bach/bachelor/playground/skraafoto_images/2023_82_23_2_0010_00001123.tif')
    rotation_matrix_list = [0.9999777436461087, -0.00661694332508786, -0.0008533894011739274,
                            0.005274316520016631, 0.7057005897569285, 0.7084905498325083,
                            -0.0040858044109012915, -0.7084792824619192, 0.7057197832883532]

img_height, img_width, _ = image.shape
polygon = Polygon(poly)
rotation_matrix = np.array(rotation_matrix_list).reshape(3, 3)

# Extract bounding box from geo-coordinates
min_lon, min_lat, max_lon, max_lat = polygon.bounds

# Define real-world control points for precise homography
src_pts = np.array([
    [0, 0],  # Top-left (Image pixel coordinates)
    [img_width - 1, 0],  # Top-right
    [img_width - 1, img_height - 1],  # Bottom-right
    [0, img_height - 1]  # Bottom-left
], dtype=np.float32)

dst_pts = np.array([
    [min_lon, max_lat],  # Corresponding Geo-coordinates
    [max_lon, max_lat],
    [max_lon, min_lat],
    [min_lon, min_lat]
], dtype=np.float32)

# Compute homography matrix
H, _ = cv2.findHomography(src_pts, dst_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
H_inv, _ = cv2.findHomography(dst_pts, src_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)

def apply_rotation_correction(x, y):
    """Applies rotation correction to pixel coordinates."""
    pixel_coords = np.array([x, y, 1], dtype=np.float32).T
    corrected_coords = np.dot(rotation_matrix, pixel_coords)
    return corrected_coords[0], corrected_coords[1]

def pixel_to_latlon(x, y):
    """Converts image pixel coordinates to latitude/longitude using homography."""
    x_corrected, y_corrected = apply_rotation_correction(x, y)
    pixel_coords = np.array([[x_corrected, y_corrected, 1]], dtype=np.float32).T
    geo_coords = np.dot(H, pixel_coords)
    geo_coords /= geo_coords[2]  # Normalize
    lat, lon = geo_coords[1][0], geo_coords[0][0]
    return round(lat, 8), round(lon, 8)

def latlon_to_pixel(lat, lon):
    """Converts latitude/longitude back to image pixel coordinates for validation."""
    geo_coords = np.array([[lon, lat, 1]], dtype=np.float32).T
    pixel_coords = np.dot(H_inv, geo_coords)
    pixel_coords /= pixel_coords[2]  # Normalize
    x, y = int(round(pixel_coords[0][0])), int(round(pixel_coords[1][0]))
    return x, y

def mouse_callback(event, x, y, flags, param):
    """Handles mouse movement over the image."""
    if event == cv2.EVENT_MOUSEMOVE:
        lat, lon = pixel_to_latlon(x, y)
        x_back, y_back = latlon_to_pixel(lat, lon)  # Validation check
        text = f"Lat: {lat:.8f}, Lon: {lon:.8f}\nX: {x}, Y: {y} (Back: {x_back}, {y_back})"
        temp_image = image.copy()
        cv2.putText(temp_image, text, (50, 400), cv2.FONT_HERSHEY_SIMPLEX, 6, (0, 255, 0), 3)
        cv2.imshow("Image", temp_image)

# Display image using OpenCV
cv2.imshow("Image", image)
cv2.setMouseCallback("Image", mouse_callback)
cv2.waitKey(0)
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [ ]:
#MAY ALLAH HELP BE FROM HERE

In [ ]:
import requests
from shapely.geometry import Polygon
import cv2
import numpy as np

local= True

if not local:
    # Load image from CDN
    response = requests.get(img_url)
    image_arr = np.array(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_arr, cv2.IMREAD_COLOR)
else:
    poly = [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]
    image = cv2.imread('/Users/papp/Documents/DomoProject/autoCheck/images/2023_82_23_2_0010_00001123.tif')

img_height, img_width, _ = image.shape

polygon = Polygon(poly)

# Get bounding box of the geojson coordinates
min_lon, min_lat, max_lon, max_lat = polygon.bounds

def pixel_to_latlon(x, y):
    """Convert image pixel coordinates to latitude/longitude with higher precision."""
    lat_range = max_lat - min_lat
    lon_range = max_lon - min_lon
    lat = max_lat - ((y + 0.5) / img_height) * lat_range  # Centering pixels for precision
    lon = min_lon + ((x + 0.5) / img_width) * lon_range
    return round(lat, 8), round(lon, 8)  # Increased decimal places for precision

def mouse_callback(event, x, y, flags, param):
    """Handle mouse movement over the image."""
    if event == cv2.EVENT_MOUSEMOVE:
        lat, lon = pixel_to_latlon(x-200, y+900)
        text = f"Lat: {lat:.6f}, Lon: {lon:.6f} X: {x}, Y: {y}"
        temp_image = image.copy()
        cv2.putText(temp_image, text, (00, 400), cv2.FONT_HERSHEY_SIMPLEX, 6, (0, 255, 0), 12)
        cv2.imshow("Image", temp_image)

# Display image using OpenCV
cv2.imshow("Image", image)
cv2.setMouseCallback("Image", mouse_callback)
cv2.waitKey(0)
cv2.destroyAllWindows()


[ WARN:0@0.012] global loadsave.cpp:268 findDecoder imread_('/Users/papp/Documents/DomoProject/autoCheck/images/2023_82_23_2_0010_00001123.tif'): can't open/read file: check file path/integrity


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
import requests
from shapely.geometry import Polygon
import cv2
import numpy as np

local= True

if not local:
    # Load image from CDN
    response = requests.get(img_url)
    image_arr = np.array(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_arr, cv2.IMREAD_COLOR)
else:
    poly = [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]
    image = cv2.imread('/Users/papp/Documents/DomoProject/autoCheck/images/2023_82_23_2_0010_00001123.tif')


img_height, img_width, _ = image.shape
print(img_height, img_width)
polygon = Polygon(poly)

# Get bounding box of the geojson coordinates
min_lon, min_lat, max_lon, max_lat = polygon.bounds

print(min_lon, min_lat, max_lon, max_lat)

# Define real-world control points for precise homography
src_pts = np.array([
    [0, 0],  # Top-left (Image pixel coordinates)
    [img_width - 1, 0],  # Top-right
    [img_width - 1, img_height - 1],  # Bottom-right
    [0, img_height - 1]  # Bottom-left
], dtype=np.float32)

dst_pts = np.array([
    [min_lon, max_lat],  # Corresponding Geo-coordinates
    [max_lon, max_lat],
    [max_lon, min_lat],
    [min_lon, min_lat]
], dtype=np.float32)

# Compute homography matrix
H, _ = cv2.findHomography(src_pts, dst_pts, method=cv2.RANSAC)

# Compute inverse homography for validation
H_inv, _ = cv2.findHomography(dst_pts, src_pts, method=cv2.RANSAC)

def pixel_to_latlon(x, y):
    """Convert image pixel coordinates to latitude/longitude using homography."""
    pixel_coords = np.array([[x, y, 1]], dtype=np.float32).T
    geo_coords = np.dot(H, pixel_coords)
    geo_coords /= geo_coords[2]  # Normalize
    lat, lon = geo_coords[1][0], geo_coords[0][0]
    return round(lat, 8), round(lon, 8)

def latlon_to_pixel(lat, lon):
    """Convert latitude/longitude back to image pixel coordinates for validation."""
    geo_coords = np.array([[lon, lat, 1]], dtype=np.float32).T
    pixel_coords = np.dot(H_inv, geo_coords)
    pixel_coords /= pixel_coords[2]  # Normalize
    x, y = int(pixel_coords[0][0]), int(pixel_coords[1][0])
    return x, y

def mouse_callback(event, x, y, flags, param):
    """Handle mouse movement over the image."""
    if event == cv2.EVENT_MOUSEMOVE:
        lat, lon = pixel_to_latlon(x-200, y+900)
        x_back, y_back = latlon_to_pixel(lat, lon)  # Validation check
        text = f"Lat: {lat:.8f}, Lon: {lon:.8f} X: {x-200}, Y: {y+900} (Back: {x_back}, {y_back})"
        temp_image = image.copy()
        cv2.putText(temp_image, text, (0, 400), cv2.FONT_HERSHEY_SIMPLEX, 6, (0, 255, 0), 12)
        cv2.imshow("Image", temp_image)

# Display image using OpenCV
cv2.imshow("Image", image)
cv2.setMouseCallback("Image", mouse_callback)
cv2.waitKey(0)
cv2.destroyAllWindows()

[ WARN:0@18.903] global loadsave.cpp:268 findDecoder imread_('/Users/papp/Documents/DomoProject/autoCheck/images/2023_82_23_2_0010_00001123.tif'): can't open/read file: check file path/integrity


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
import cv2
import requests
import json
import numpy as np
from shapely.geometry import Polygon
from io import BytesIO
from pyproj import Transformer
from PIL import Image

# Define the image CDN URL and GeoJSON file path
IMAGE_URL = "https://your-cdn-url.com/sample.jpg"
GEOJSON_FILE = "boundary.json"  # Update with the actual GeoJSON file path

local= True

if not local:
    # Load image from CDN
    response = requests.get(img_url)
    image_arr = np.array(bytearray(response.content), dtype=np.uint8)
    image = cv2.imdecode(image_arr, cv2.IMREAD_COLOR)
else:
    poly = [[9.791136734961823, 55.88267537764091], [9.816435520740168, 55.88258317504471], [9.81245077005829, 55.86958349197774], [9.79422746331042, 55.86973171184553], [9.791136734961823, 55.88267537764091]]
    image = cv2.imread('/Users/papp/Documents/DomoProject/autoCheck/images/2023_82_23_2_0010_00001123.tif')
    rotation_matrix_list= [0.9999777436461087, -0.00661694332508786, -0.0008533894011739274, 0.005274316520016631, 0.7057005897569285, 0.7084905498325083, -0.0040858044109012915, -0.7084792824619192, 0.7057197832883532]

img_height, img_width, _ = image.shape
print(img_height, img_width)
polygon = Polygon(poly)

# Get bounding box of the geojson coordinates
min_lon, min_lat, max_lon, max_lat = polygon.bounds

# Projection metadata
proj_epsg = 25832  # ETRS89 / UTM zone 32N
perspective_center = np.array([550261.042, 6190290.415, 2184.795])  # X, Y, Z in UTM
omega, phi, kappa = np.radians([45.1118, -0.2341, -0.3022])  # Convert angles to radians

# Create a transformer for UTM to WGS84
transformer = Transformer.from_crs(f"EPSG:{proj_epsg}", "EPSG:4326", always_xy=True)

def rotation_matrix(omega, phi, kappa):
    """Calculate the rotation matrix from omega, phi, kappa angles."""
    R_x = np.array([[1, 0, 0], [0, np.cos(omega), -np.sin(omega)], [0, np.sin(omega), np.cos(omega)]])
    R_y = np.array([[np.cos(phi), 0, np.sin(phi)], [0, 1, 0], [-np.sin(phi), 0, np.cos(phi)]])
    R_z = np.array([[np.cos(kappa), -np.sin(kappa), 0], [np.sin(kappa), np.cos(kappa), 0], [0, 0, 1]])
    return R_z @ R_y @ R_x  # Final rotation matrix

R = rotation_matrix(omega, phi, kappa)
R_inv = np.linalg.inv(R)  # Compute inverse rotation matrix

def pixel_to_latlon(x, y):
    """Convert image pixel coordinates to latitude/longitude using the rotation matrix."""
    norm_x = (x - img_width / 2) / img_width * 10000  # Scale factor for demo
    norm_y = (y - img_height / 2) / img_height * 10000
    norm_z = 0  # Assuming the image is on a flat plane

    pixel_vector = np.array([norm_x, norm_y, norm_z])
    rotated_vector = R @ pixel_vector
    utm_coords = perspective_center[:2] + rotated_vector[:2]
    lon, lat = transformer.transform(utm_coords[0], utm_coords[1])
    return lat, lon

def latlon_to_pixel(lat, lon):
    """Convert latitude/longitude back to image pixel coordinates using inverse rotation matrix."""
    utm_x, utm_y = transformer.transform(lon, lat, direction="INVERSE")
    utm_vector = np.array([utm_x - perspective_center[0], utm_y - perspective_center[1], 0])
    pixel_vector = R_inv @ utm_vector
    x = int((pixel_vector[0] / 10000) * img_width + img_width / 2)
    y = int((pixel_vector[1] / 10000) * img_height + img_height / 2)
    return x, y

def mouse_callback(event, x, y, flags, param):
    """Handle mouse movement over the image."""
    if event == cv2.EVENT_MOUSEMOVE:
        lat, lon = pixel_to_latlon(x, y)
        x_back, y_back = latlon_to_pixel(lat, lon)  # Apply inverse rotation
        img_copy = image.copy()
        text = f"Lat: {lat:.6f}, Lon: {lon:.6f}, X: {x}, Y: {y}, Back: ({x_back}, {y_back})"
        cv2.putText(img_copy, text, (10, 400), cv2.FONT_HERSHEY_SIMPLEX, 6, (0, 255, 0), 12)
        cv2.imshow("Image to Lat/Lon Converter", img_copy)

# Display image with OpenCV
cv2.namedWindow("Image to Lat/Lon Converter")
cv2.setMouseCallback("Image to Lat/Lon Converter", mouse_callback)
cv2.imshow("Image to Lat/Lon Converter", image)
cv2.waitKey(0)
cv2.destroyAllWindows()


10560 14144


: 

: 